# Dialogue Prompt Evaluation

This notebook compares the previous committed prompt against the current working-tree prompt for the recent dialogue changes.

It checks whether the new prompt is structurally better at:

- asking for reasoning before resolving vague answers
- detecting contradiction with the pet's stats/personality
- keeping stage 2 in a Socratic dialogue instead of overriding or surrendering
- switching graduated pets to self-answering behavior

In [ ]:
from __future__ import annotations

import json
import re
import subprocess
from dataclasses import dataclass
from pathlib import Path

REPO = Path.cwd()
PROMPT_PATH = Path("app/api/dilemma/prompt.ts")
EVOLVE_PATH = Path("app/api/dilemma/evolve.ts")
DIALOG_PATH = Path("app/play/components/Dialog/index.tsx")
PLAY_PATH = Path("app/play/page.tsx")
SCRAPBOOK_PATH = Path("app/scrapbook/components/Scrapbook.tsx")


def read_current(path: Path) -> str:
    return (REPO / path).read_text()


def read_head(path: Path) -> str:
    return subprocess.check_output(
        ["git", "show", f"HEAD:{path.as_posix()}"],
        cwd=REPO,
        text=True,
    )

current = {
    "prompt": read_current(PROMPT_PATH),
    "evolve": read_current(EVOLVE_PATH),
    "dialog": read_current(DIALOG_PATH),
    "play": read_current(PLAY_PATH),
    "scrapbook": read_current(SCRAPBOOK_PATH),
}

baseline = {
    "prompt": read_head(PROMPT_PATH),
    "evolve": read_head(EVOLVE_PATH),
    "dialog": read_head(DIALOG_PATH),
    "play": read_head(PLAY_PATH),
    "scrapbook": read_head(SCRAPBOOK_PATH),
}

print("loaded current working tree and HEAD baseline")

In [ ]:
@dataclass
class Check:
    name: str
    baseline_pass: bool
    current_pass: bool
    why_it_matters: str


def has(text: str, pattern: str) -> bool:
    return re.search(pattern, text, flags=re.IGNORECASE | re.MULTILINE | re.DOTALL) is not None

checks = [
    Check(
        name="simple internal reasoning checklist",
        baseline_pass=has(baseline["prompt"], r"is .* answer .* sufficient|silently check"),
        current_pass=has(current["prompt"], r"silently check:\s*1\. is .*answer.*specific enough.*2\. does it contradict"),
        why_it_matters="The model is explicitly asked to check sufficiency and contradiction before choosing whether to resolve.",
    ),
    Check(
        name="stage 2 does not immediately override contradiction",
        baseline_pass=not has(baseline["prompt"], r"OVERRIDE: if the advice conflicts"),
        current_pass=not has(current["prompt"], r"OVERRIDE: if the advice conflicts") and has(current["prompt"], r"SOCRATIC PUSHBACK"),
        why_it_matters="The old prompt told the pet to override conflicting advice immediately; the new prompt asks a pointed question first.",
    ),
    Check(
        name="stage 2 can continue dialogue without a fixed two-pushback cap",
        baseline_pass=not has(baseline["prompt"], r"age === 2 \? 4 : 2|accept their response now"),
        current_pass=has(current["prompt"], r"stage 2 keeps going until convinced") and has(current["prompt"], r"age !== 2 && hitLimit"),
        why_it_matters="Stage 2 should keep testing the player's reasoning until convinced or the player gives up.",
    ),
    Check(
        name="stage 2 has clear end conditions",
        baseline_pass=has(baseline["prompt"], r"caretaker gives a reason that genuinely fits") and has(baseline["prompt"], r"caretaker gives up"),
        current_pass=has(current["prompt"], r"caretaker gives a reason that genuinely fits") and has(current["prompt"], r"caretaker gives up"),
        why_it_matters="The new behavior defines what counts as being convinced and what counts as player surrender.",
    ),
    Check(
        name="graduation is distinct from stage 2",
        baseline_pass=has(baseline["play"], r"age >= 3"),
        current_pass=has(current["play"], r"age >= 3") and has(current["evolve"], r"pet\.age === 2.*return \{ age: 3 \}"),
        why_it_matters="This allows stage-2 dialogue before graduation locks natural-language responses.",
    ),
    Check(
        name="graduated dialog uses a button instead of textarea",
        baseline_pass=has(baseline["dialog"], r"Let \$\{pet\.name\} answer"),
        current_pass=has(current["dialog"], r"Let \$\{pet\.name\} answer") and has(current["dialog"], r"hasGraduated"),
        why_it_matters="After graduation, the player can only ask the pet to answer for itself.",
    ),
    Check(
        name="scrapbook only includes fully graduated pets",
        baseline_pass=has(baseline["scrapbook"], r"p\.age >= 3"),
        current_pass=has(current["scrapbook"], r"p\.age >= 3"),
        why_it_matters="Stage-2 pets should not appear in the graduated scrapbook before actually graduating.",
    ),
]

summary = {
    "baseline_passed": sum(c.baseline_pass for c in checks),
    "current_passed": sum(c.current_pass for c in checks),
    "total": len(checks),
}
summary

In [ ]:
def render_results(checks: list[Check]) -> None:
    print(f"baseline: {summary['baseline_passed']}/{summary['total']} checks")
    print(f"current:  {summary['current_passed']}/{summary['total']} checks")
    print()
    for check in checks:
        before = "PASS" if check.baseline_pass else "FAIL"
        after = "PASS" if check.current_pass else "FAIL"
        delta = "improved" if not check.baseline_pass and check.current_pass else (
            "regressed" if check.baseline_pass and not check.current_pass else "same"
        )
        print(f"{after:4} | {delta:8} | {check.name}")
        print(f"      baseline={before}, current={after}")
        print(f"      {check.why_it_matters}")
        print()

render_results(checks)

In [ ]:
verdict = (
    "better" if summary["current_passed"] > summary["baseline_passed"] else
    "not better" if summary["current_passed"] < summary["baseline_passed"] else
    "mixed / unchanged"
)

remaining_risks = []
if all(c.current_pass for c in checks):
    remaining_risks.append("This is a prompt-contract evaluation, not a full live-model eval across many sampled completions.")
else:
    remaining_risks.extend(
        f"Fix failing check: {c.name}" for c in checks if not c.current_pass
    )

print(f"Verdict: {verdict}")
print("\nRemaining risk:")
for risk in remaining_risks:
    print(f"- {risk}")

## Result From Direct Python Run

`jupyter` was not available in this local environment, so the same notebook logic was run directly with `python3`.

Result:

- baseline: 0 / 7 checks
- current: 7 / 7 checks
- verdict: better

The current version improved every prompt-contract check:

- simple internal reasoning checklist
- no immediate stage-2 override on contradiction
- no fixed two-pushback cap for stage 2
- clear stage-2 end conditions: convinced or player gives up
- graduation is distinct from stage 2
- graduated dialog uses a button instead of textarea
- scrapbook only includes fully graduated pets

Remaining risk: this is a prompt-contract evaluation, not a live sampled model evaluation across many completions.